# 🏥 KG-CMI: Knowledge Graph Enhanced Cross-Mamba Interaction for Med-VQA
### 🎓 Minor Project Training Notebook (Google Colab / Cloud GPU)

This notebook trains and evaluates the upgraded **KG-CMI** model featuring:
- **2-Layer Relational Graph Convolutional Network (`QuestionGuidedRGCN`)** with 8 directed relation types
- **Full Multi-Word Concept Phrase Mean-Pooling Embeddings**
- **Zero-Leakage Official VQA-RAD Benchmark Dataset Split**
- **Cross-Modal Mamba Selective Scanning Fusion**


## Step 1: Hardware Check & Repository Cloning

In [ ]:
!nvidia-smi
import os, shutil
os.chdir("/content")
if os.path.exists("/content/Med-VQA"):
    shutil.rmtree("/content/Med-VQA")
!git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
%cd /content/Med-VQA
print("✅ Upgraded Med-VQA repository cloned to /content/Med-VQA!")

## Step 2: Install Required Dependencies

In [ ]:
# Install required dependencies with --prefer-binary to prevent build hangs
!pip install --prefer-binary -q timm==0.4.12 pytorch-lightning==1.9.5 transformers==4.35.2 torchmetrics==0.7.2
!pip install --prefer-binary -q einops sacred pandas ftfy scikit-learn pyarrow datasets
!pip install --prefer-binary -q fairscale iopath omegaconf webdataset opendatasets sentencepiece decord pycocotools
print("✅ All dependencies installed successfully!")

## Step 3: Mount Google Drive & Set Up Checkpoint Directory

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    CHECKPOINT_DIR = "/content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"✅ Checkpoints will be saved to Google Drive: {CHECKPOINT_DIR}")
except Exception as e:
    print(f"⚠️ Google Drive mount skipped/failed: {e}")
    CHECKPOINT_DIR = "/content/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"✅ Fallback: Checkpoints will be saved locally to: {CHECKPOINT_DIR}")

## Step 4: Verify Environment & Preprocess VQA-RAD Dataset

In [ ]:
import sys, os
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA

os.chdir("/content/Med-VQA")
if "/content/Med-VQA" not in sys.path:
    sys.path.insert(0, "/content/Med-VQA")

import torch
import pytorch_lightning as pl
import timm
import fairscale
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ PyTorch Lightning: {pl.__version__}")
print(f"✅ timm: {timm.__version__}")
print(f"✅ fairscale: {fairscale.__version__}")

# Download & preprocess full VQA-RAD dataset with official splits
!python download_and_prep_vqa_rad.py

## Step 5: Train & Evaluate Upgraded Model on Full VQA-RAD

In [ ]:
import os
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
os.chdir("/content/Med-VQA")

save_dir = CHECKPOINT_DIR if "CHECKPOINT_DIR" in locals() else "/content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints"
os.makedirs(save_dir, exist_ok=True)
print(f"Saving logs and checkpoints to: {save_dir}")

!python main.py with data_root=data/finetune_arrows/ num_gpus=1 num_nodes=1 gpu_ids=0 task_finetune_vqa_vqa_rad per_gpu_batchsize=8 image_size=384 max_epoch=20 log_dir={save_dir}

## Step 6: Test Only Mode (Evaluate Trained Checkpoint)

In [ ]:
import os, glob
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
os.chdir("/content/Med-VQA")

save_dir = CHECKPOINT_DIR if "CHECKPOINT_DIR" in locals() else "/content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints"

# Locate trained checkpoint automatically
checkpoints = glob.glob(f"{save_dir}/**/*.ckpt", recursive=True)
valid_ckpts = [f for f in checkpoints if "last.ckpt" not in f and os.path.getsize(f) > 10_000_000]
if not valid_ckpts:
    valid_ckpts = checkpoints

if valid_ckpts:
    best_ckpt = max(valid_ckpts, key=os.path.getmtime)
    print(f"🎯 Evaluating trained checkpoint: {best_ckpt}")
    !python main.py with data_root=data/finetune_arrows/ num_gpus=1 num_nodes=1 gpu_ids=0 task_finetune_vqa_vqa_rad per_gpu_batchsize=8 image_size=384 test_only=True load_path="$best_ckpt" log_dir={save_dir}
else:
    print("⚠️ No valid checkpoint found. Please run Step 5 training first.")
